# Berkeley Open Data Analysis Pipeline
## Integration with Datasette, Pandas, NumPy, TensorFlow, Plotly, and Seaborn

**Created:** 2025-01-13  and Jan 6, 2026

**Purpose:** Fetch, analyze, and visualize City of Berkeley Open Data

### Workflow:
1. Connect to Berkeley Open Data API (Socrata)
2. Fetch Business Licenses and other datasets
3. Clean and process data
4. Export to CSV, JSON, GeoJSON
5. Load into Datasette (SQLite)
6. Analyze with Pandas/NumPy
7. Visualize with Plotly/Seaborn
8. Optional: ML with TensorFlow

In [2]:
# CELL 1: Install Required Packages
# Run this cell once to install all dependencies

%pip install pandas numpy geopandas sodapy datasette plotly seaborn tensorflow folium requests

print("✅ Installation newly complete!")

Note: you may need to restart the kernel to use updated packages.
✅ Installation newly complete!


In [3]:
# CELL 2: Import Libraries

import pandas as pd
import numpy as np
import geopandas as gpd
from sodapy import Socrata
import json
import sqlite3
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✅ All libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

✅ All libraries imported successfully
Pandas version: 2.3.3
NumPy version: 2.3.5


In [19]:
# CELL 3: Configure Berkeley Open Data API - new
# CELL 3: Configure Berkeley Open Data API

# 🔑 IMPORTANT: You Need a Free App Token!
# 
# Berkeley's API requires authentication. Get your FREE token here:
# https://data.cityofberkeley.info/profile/edit/developer_settings
# 
# It takes 2 minutes and is completely free!
# 
# Quick Setup:
# 1. Create account at https://data.cityofberkeley.info
# 2. Go to Developer Settings
# 3. Create New App Token
# 4. Copy your token
# 5. Either:
#    - Create .env file with: BERKELEY_APP_TOKEN=your-token
#    - OR paste token directly below
# 
# See TOKEN_SETUP_INSTRUCTIONS.md for detailed help!

import os

# Try to load from environment variable first
try:
    from dotenv import load_dotenv
    load_dotenv()
    print("✅ Loaded .env file")
except:
    print("ℹ️  python-dotenv not installed (optional)")

# Berkeley Open Data Portal Configuration
BERKELEY_DOMAIN = "data.cityofberkeley.info"

# 🔑 SET YOUR APP TOKEN HERE
# Get your free token from: https://data.cityofberkeley.info/profile/edit/developer_settings

# Option 1: From environment variable (recommended - keeps token private)
APP_TOKEN = os.environ.get('BERKELEY_APP_TOKEN')

# Option 2: Paste directly here (quick, but less secure if sharing)
# Uncomment the line below and add your token:
APP_TOKEN = "sKTwtTUlhd2VfrmCC9W3xKr9P"  # Replace with your actual token

# Check if token is set
if APP_TOKEN:
    print(f"✅ App token loaded: {APP_TOKEN[:8]}...")
else:
    print("\n" + "="*70)
    print("⚠️  WARNING: No app token found!")
    print("="*70)
    print("\nYou need a FREE app token to access Berkeley's data.")
    print("\n📝 Quick Setup (2 minutes):")
    print("   1. Go to: https://data.cityofberkeley.info")
    print("   2. Sign up (free)")
    print("   3. Developer Settings → Create App Token")
    print("   4. Copy token and either:")
    print("      - Create .env file with: BERKELEY_APP_TOKEN=your-token")
    print("      - Or paste directly in this cell (see Option 2 above)")
    print("\n📖 See TOKEN_SETUP_INSTRUCTIONS.md for detailed help")
    print("="*70)
    print("\nℹ️  Continuing without token - may get 403 errors...")

# CORRECTED Dataset IDs
DATASETS = {
    'business_licenses': 'rwnf-bu3w',  # ✅ CORRECT ID
    'crime_incidents': 'k2nh-s5h5',
    'restaurant_inspections': 'b47j-kakm',
    'building_permits': 'ydr8-5enu',
    'building contract': ' kvz2-j5cj',
    'DOB Permit Issuance': 'ipu4-2q9a',
    'Affordable Rental Housing' : 's6ha-ppgi',
    'Issued Construction Permits': '3syk-w9eu'
}

# Initialize Socrata client
from sodapy import Socrata

client = Socrata(BERKELEY_DOMAIN, APP_TOKEN)

print(f"\n✅ Connected to {BERKELEY_DOMAIN}")
if APP_TOKEN:
    print("   Using app token (10,000 requests/hour)")
else:
    print("   ⚠️  No token - limited to 1,000 requests/hour and may be blocked")
print(f"\n📊 Available datasets: {list(DATASETS.keys())}")



✅ Loaded .env file
✅ App token loaded: sKTwtTUl...

✅ Connected to data.cityofberkeley.info
   Using app token (10,000 requests/hour)

📊 Available datasets: ['business_licenses', 'crime_incidents', 'restaurant_inspections', 'building_permits', 'building contract', 'DOB Permit Issuance', 'Affordable Rental Housing', 'Issued Construction Permits']


## try getting new datasets from Berkeley

In [7]:
# Cell to get lists of datasets
# CELL: Find Berkeley Dataset IDs

import requests
import pandas as pd

DOMAIN = "data.cityofberkeley.info"
APP_TOKEN = "sKTwtTUlhd2VfrmCC9W3xKr9P"  # Your actual token

catalog_url = f"https://{DOMAIN}/api/catalog/v1"

print("🔍 SEARCHING BERKELEY OPEN DATA CATALOG...\n")
print("="*70)

headers = {'X-App-Token': APP_TOKEN}

try:
    response = requests.get(catalog_url, headers=headers, params={'limit': 500}, timeout=30)
    
    if response.status_code == 200:
        catalog = response.json()
        results = catalog.get('results', [])
        
        print(f"Found {len(results)} total datasets\n")
        
        # Extract dataset info
        datasets = []
        
        for item in results:
            resource = item.get('resource', {})
            datasets.append({
                'id': resource.get('id', 'N/A'),
                'name': resource.get('name', 'N/A'),
                'type': resource.get('type', 'N/A'),
                'description': resource.get('description', '')[:100]
            })
        
        # Create DataFrame
        df = pd.DataFrame(datasets)
        
        # Filter for permit/building/construction related
        keywords = ['permit', 'building', 'construction', 'zoning', 'development', 'housing']
        
        mask = df['name'].str.contains('|'.join(keywords), case=False, na=False) | \
               df['description'].str.contains('|'.join(keywords), case=False, na=False)
        
        relevant = df[mask]
        
        print(f"📋 PERMIT/BUILDING RELATED DATASETS ({len(relevant)}):\n")
        
        for idx, row in relevant.iterrows():
            print(f"{row['name']}")
            print(f"   ID: {row['id']}")
            print(f"   Type: {row['type']}")
            print(f"   URL: https://{DOMAIN}/resource/{row['id']}.json")
            print()
        
        # Save all to CSV for reference
        df.to_csv('/Users/johngage/berkeley-data/berkeley_datasets.csv', index=False)
        print(f"✅ Saved all {len(df)} datasets to: berkeley_datasets.csv")
        
    else:
        print(f"❌ Error: {response.status_code}")
        print(response.text[:500])
        
except Exception as e:
    print(f"❌ Error: {e}")

print("\n" + "="*70)


🔍 SEARCHING BERKELEY OPEN DATA CATALOG...

Found 500 total datasets

📋 PERMIT/BUILDING RELATED DATASETS (20):

Building, Electrical, Fire, Grading, Mechanical, Plumbing & Sign Permits
   ID: kvz2-j5cj
   Type: dataset
   URL: https://data.cityofberkeley.info/resource/kvz2-j5cj.json

Building Permits
   ID: ydr8-5enu
   Type: dataset
   URL: https://data.cityofberkeley.info/resource/ydr8-5enu.json

DOB Permit Issuance
   ID: ipu4-2q9a
   Type: dataset
   URL: https://data.cityofberkeley.info/resource/ipu4-2q9a.json

Affordable Rental Housing Developments
   ID: s6ha-ppgi
   Type: dataset
   URL: https://data.cityofberkeley.info/resource/s6ha-ppgi.json

Issued Construction Permits
   ID: 3syk-w9eu
   Type: dataset
   URL: https://data.cityofberkeley.info/resource/3syk-w9eu.json

Film Permits
   ID: tg4x-b46p
   Type: dataset
   URL: https://data.cityofberkeley.info/resource/tg4x-b46p.json

County Building Codes for Missouri
   ID: iq7s-izvt
   Type: dataset
   URL: https://data.cityofber

In [12]:
# CELL: Investigate Domain Values

import pandas as pd

# Load the original unfiltered data
df = pd.read_csv('/Users/johngage/berkeley-data/berkeley_datasets.csv')

print("🔍 INVESTIGATING DATASET DOMAINS...\n")
print("="*70)

print(f"\nTotal datasets: {len(df)}")

# Check what columns exist
print(f"\nColumns: {df.columns.tolist()}")

# If 'domain' column exists, show unique values
if 'domain' in df.columns:
    print(f"\n📊 Unique domains ({df['domain'].nunique()}):")
    print(df['domain'].value_counts().head(20))
else:
    print("\n⚠️  No 'domain' column found")

# If 'permalink' exists, show some examples
if 'permalink' in df.columns:
    print(f"\n📊 Sample permalinks:")
    print(df['permalink'].head(10).tolist())
else:
    print("\n⚠️  No 'permalink' column found")

# Check the 'id' patterns
print(f"\n📊 Sample dataset IDs:")
print(df['id'].head(20).tolist())

# Check 'name' column for Berkeley mentions
if 'name' in df.columns:
    berkeley_in_name = df['name'].str.contains('berkeley', case=False, na=False).sum()
    print(f"\n📊 Datasets with 'Berkeley' in name: {berkeley_in_name}")
    
    if berkeley_in_name > 0:
        print("\nExamples:")
        print(df[df['name'].str.contains('berkeley', case=False, na=False)]['name'].head(10).tolist())

print("\n" + "="*70)

🔍 INVESTIGATING DATASET DOMAINS...


Total datasets: 500

Columns: ['id', 'name', 'type', 'description']

⚠️  No 'domain' column found

⚠️  No 'permalink' column found

📊 Sample dataset IDs:
['9fxf-t2tr', 'kwxv-fwze', 'q5as-kyim', '9bhg-hcku', 'a7mk-8suc', 'qxh8-f4bd', 'rxn6-qnx8', 'qccx-65fg', '8wbx-tsch', 'vx8i-nprf', '6rkd-2wt9', 'mu99-t4jn', 'jixs-h7uw', 'wa3g-tfvc', 'juse-v5tw', 'vduf-at7y', 'keti-qx5t', 'j4qr-keke', 'pueh-98ic', 'ic3t-wcy2']

📊 Datasets with 'Berkeley' in name: 0



In [ ]:
# CELL: Discover All Berkeley Datasets: NO NO

import requests
import pandas as pd
import json

DOMAIN = "data.cityofberkeley.info"
APP_TOKEN = "sKTwtTUlhd2VfrmCC9W3xKr9P"

print("🔍 DISCOVERING BERKELEY DATASETS...\n")
print("="*70)

headers = {'X-App-Token': APP_TOKEN}
catalog_url = f"https://{DOMAIN}/api/catalog/v1"

try:
    # Get catalog
    response = requests.get(
        catalog_url, 
        headers=headers, 
        params={'limit': 2000, 'only': 'datasets'},  # Only datasets, not charts
        timeout=30
    )
    
    if response.status_code == 200:
        catalog = response.json()
        results = catalog.get('results', [])
        
        print(f"Found {len(results)} items in catalog\n")
        
        # Extract relevant info
        datasets = []
        for item in results:
            resource = item.get('resource', {})
            classification = item.get('classification', {})
            metadata = item.get('metadata', {})
            
            datasets.append({
                'id': resource.get('id', 'N/A'),
                'name': resource.get('name', 'N/A'),
                'description': resource.get('description', '')[:200],
                'type': resource.get('type', 'N/A'),
                'category': classification.get('categories', []),
                'tags': classification.get('tags', []),
                'updated': metadata.get('rowsUpdatedAt', 'N/A')
            })
        
        df = pd.DataFrame(datasets)
        
        # Save all datasets
        df.to_csv('/Users/johngage/berkeley-data/all_berkeley_datasets.csv', index=False)
        print(f"✅ Saved all {len(df)} datasets to: all_berkeley_datasets.csv\n")
        
        # Filter for building/housing/energy/water related
        keywords = [
            'permit', 'building', 'construction', 'zoning', 'housing', 
            'energy', 'water', 'utility', 'residential', 'development',
            'inspection', 'code', 'compliance'
        ]
        
        def matches_keywords(row):
            text = f"{row['name']} {row['description']} {row['category']} {row['tags']}".lower()
            return any(kw in text for kw in keywords)
        
        relevant = df[df.apply(matches_keywords, axis=1)]
        
        print(f"📋 RELEVANT DATASETS ({len(relevant)}):\n")
        print("="*70)
        
        for idx, row in relevant.iterrows():
            print(f"\n{row['name']}")
            print(f"   ID: {row['id']}")
            print(f"   Type: {row['type']}")
            print(f"   Description: {row['description'][:100]}...")
            print(f"   CSV: https://{DOMAIN}/resource/{row['id']}.csv")
            print(f"   JSON: https://{DOMAIN}/resource/{row['id']}.json")
        
        # Save relevant datasets
        relevant.to_csv('/Users/johngage/berkeley-data/relevant_datasets.csv', index=False)
        print(f"\n✅ Saved {len(relevant)} relevant datasets to: relevant_datasets.csv")
        
    else:
        print(f"❌ Error {response.status_code}: {response.text[:200]}")
        
except Exception as e:
    print(f"❌ Error: {e}")

print("\n" + "="*70)

🔍 DISCOVERING BERKELEY DATASETS...

Found 2000 items in catalog

✅ Saved all 2000 datasets to: all_berkeley_datasets.csv

📋 RELEVANT DATASETS (338):


Building, Electrical, Fire, Grading, Mechanical, Plumbing & Sign Permits
   ID: kvz2-j5cj
   Type: dataset
   Description: Issued Permits - File Date:  5/23/2013 - 11/30/2025...
   CSV: https://data.cityofberkeley.info/resource/kvz2-j5cj.csv
   JSON: https://data.cityofberkeley.info/resource/kvz2-j5cj.json

Building Permits
   ID: ydr8-5enu
   Type: dataset
   Description: <b>Note, 10/15/2025:</b> We have added a PERMIT_CONDITION column.

This dataset includes information...
   CSV: https://data.cityofberkeley.info/resource/ydr8-5enu.csv
   JSON: https://data.cityofberkeley.info/resource/ydr8-5enu.json

Food Service Establishment: Last Inspection
   ID: cnih-y5dw
   Type: dataset
   Description: This data includes the name and location of food service establishments and the violations that were...
   CSV: https://data.cityofberkeley.info

In [14]:
# CELL: Get ONLY Berkeley Datasets (Direct Portal Method) SCRAPING

import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

DOMAIN = "data.cityofberkeley.info"
APP_TOKEN = "sKTwtTUlhd2VfrmCC9W3xKr9P"

print("🔍 SCRAPING BERKELEY DATA PORTAL...\n")
print("="*70)

# Berkeley's browse page
browse_url = f"https://{DOMAIN}/browse"

try:
    response = requests.get(browse_url, timeout=30)
    
    if response.status_code == 200:
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Find dataset links (format: /d/xxxx-xxxx)
        dataset_ids = set()
        
        for link in soup.find_all('a', href=True):
            href = link['href']
            # Match /d/xxxx-xxxx pattern
            match = re.search(r'/d/([a-z0-9]{4}-[a-z0-9]{4})', href)
            if match:
                dataset_ids.add(match.group(1))
        
        print(f"Found {len(dataset_ids)} unique dataset IDs\n")
        
        # Now test each one
        datasets = []
        headers = {'X-App-Token': APP_TOKEN}
        
        for dataset_id in sorted(dataset_ids):
            print(f"Testing {dataset_id}...", end=' ')
            
            # Get metadata
            metadata_url = f"https://{DOMAIN}/api/views/{dataset_id}.json"
            
            try:
                meta_response = requests.get(metadata_url, headers=headers, timeout=10)
                
                if meta_response.status_code == 200:
                    meta = meta_response.json()
                    
                    name = meta.get('name', 'N/A')
                    description = meta.get('description', '')
                    category = meta.get('category', '')
                    tags = meta.get('tags', [])
                    
                    datasets.append({
                        'id': dataset_id,
                        'name': name,
                        'description': description[:200],
                        'category': category,
                        'tags': ', '.join(tags) if tags else ''
                    })
                    
                    print(f"✅ {name[:50]}")
                else:
                    print(f"❌ {meta_response.status_code}")
                    
            except Exception as e:
                print(f"❌ {str(e)[:30]}")
        
        # Save all Berkeley datasets
        df = pd.DataFrame(datasets)
        df.to_csv('/Users/johngage/berkeley-data/berkeley_datasets_REAL.csv', index=False)
        
        print(f"\n✅ Found {len(df)} real Berkeley datasets")
        print(f"✅ Saved to: berkeley_datasets_REAL.csv\n")
        
        # Filter for relevant ones
        keywords = [
            'permit', 'building', 'construction', 'zoning', 'housing', 
            'energy', 'water', 'utility', 'residential', 'development'
        ]
        
        def is_relevant(row):
            text = f"{row['name']} {row['description']} {row['category']} {row['tags']}".lower()
            return any(kw in text for kw in keywords)
        
        relevant = df[df.apply(is_relevant, axis=1)]
        
        print(f"📋 RELEVANT BERKELEY DATASETS ({len(relevant)}):\n")
        print("="*70)
        
        for idx, row in relevant.iterrows():
            print(f"\n{row['name']}")
            print(f"   ID: {row['id']}")
            print(f"   Category: {row['category']}")
            print(f"   CSV: https://{DOMAIN}/resource/{row['id']}.csv")
            print(f"   JSON: https://{DOMAIN}/resource/{row['id']}.json")
        
        relevant.to_csv('/Users/johngage/berkeley-data/berkeley_relevant_REAL.csv', index=False)
        print(f"\n✅ Saved {len(relevant)} relevant datasets")
        
    else:
        print(f"❌ Could not access Berkeley portal: {response.status_code}")
        
except Exception as e:
    print(f"❌ Error: {e}")

print("\n" + "="*70)

🔍 SCRAPING BERKELEY DATA PORTAL...

Found 0 unique dataset IDs


✅ Found 0 real Berkeley datasets
✅ Saved to: berkeley_datasets_REAL.csv

📋 RELEVANT BERKELEY DATASETS (0):


✅ Saved 0 relevant datasets



In [26]:
# CELL: Test Known Berkeley Building/Housing Datasets

import requests
import pandas as pd

DOMAIN = "data.cityofberkeley.info"
APP_TOKEN = "sKTwtTUlhd2VfrmCC9W3xKr9P"

# Known Berkeley dataset IDs from earlier searches
known_berkeley_ids = [
    'ydr8-5enu',  # Building Permits
    'c2es-76ed',  # Building Permits (alternate)
    '3syk-w9eu',  # Issued Construction Permits
    'kvz2-j5cj',  # Building, Electrical, Fire, etc Permits
    'cnih-y5dw',  # Food Service Establishments (has addresses)
    'rwnf-bu3w',
    'k2nh-s5h5',
    'b47j-kakm',
    'kvz2-j5cj',
    's6ha-ppgi',
]
DATASETS = {
    'business_licenses': 'rwnf-bu3w',  # ✅ CORRECT ID
    'crime_incidents': 'k2nh-s5h5',
    'restaurant_inspections': 'b47j-kakm',
    'building_permits': 'ydr8-5enu',
    'building contract': ' kvz2-j5cj',
    'DOB Permit Issuance': 'ipu4-2q9a',
    'Affordable Rental Housing' : 's6ha-ppgi',
    'Issued Construction Permits': '3syk-w9eu'
}

print("🧪 TESTING KNOWN BERKELEY DATASETS...\n")
print("="*70)

headers = {'X-App-Token': APP_TOKEN}
working = []

for dataset_id in known_berkeley_ids:
    print(f"\n📊 Testing: {dataset_id}")
    
    # Get metadata
    metadata_url = f"https://{DOMAIN}/api/views/{dataset_id}.json"
    
    try:
        meta_response = requests.get(metadata_url, headers=headers, timeout=10)
        
        if meta_response.status_code == 200:
            meta = meta_response.json()
            name = meta.get('name', 'Unknown')
            
            print(f"   Name: {name}")
            
            # Test data access
            data_url = f"https://{DOMAIN}/resource/{dataset_id}.json"
            data_response = requests.get(data_url, headers=headers, params={'$limit': 5}, timeout=10)
            
            if data_response.status_code == 200:
                data = data_response.json()
                
                if data:
                    df = pd.DataFrame(data)
                    print(f"   ✅ ACCESSIBLE - {len(df)} sample rows, {len(df.columns)} columns")
                    print(f"   Columns: {df.columns.tolist()}")
                    
                    working.append({
                        'id': dataset_id,
                        'name': name,
                        'rows': len(df),
                        'columns': len(df.columns),
                        'csv_url': f"https://{DOMAIN}/resource/{dataset_id}.csv",
                        'json_url': f"https://{DOMAIN}/resource/{dataset_id}.json"
                    })
                else:
                    print(f"   ⚠️  Empty dataset")
            elif data_response.status_code == 403:
                print(f"   ❌ 403 Forbidden - Restricted")
            else:
                print(f"   ❌ Error: {data_response.status_code}")
        else:
            print(f"   ❌ Metadata error: {meta_response.status_code}")
            
    except Exception as e:
        print(f"   ❌ Error: {str(e)[:50]}")

print("\n" + "="*70)
print(f"\n✅ WORKING DATASETS: {len(working)}\n")

if working:
    for ds in working:
        print(f"• {ds['name']}")
        print(f"  ID: {ds['id']}")
        print(f"  CSV: {ds['csv_url']}")
        print()
    
    pd.DataFrame(working).to_csv('/Users/johngage/berkeley-data/working_berkeley_datasets.csv', index=False)
    print("✅ Saved to: working_berkeley_datasets.csv")

print("\n" + "="*70)

🧪 TESTING KNOWN BERKELEY DATASETS...


📊 Testing: ydr8-5enu
   ❌ Metadata error: 403

📊 Testing: c2es-76ed
   ❌ Metadata error: 403

📊 Testing: 3syk-w9eu
   ❌ Metadata error: 403

📊 Testing: kvz2-j5cj
   ❌ Metadata error: 403

📊 Testing: cnih-y5dw
   ❌ Metadata error: 403

📊 Testing: rwnf-bu3w
   Name: Business Licenses
   ✅ ACCESSIBLE - 5 sample rows, 19 columns
   Columns: ['apn', 'recordid', 'busdesc', 'b1_per_sub_type', 'dba', 'naics', 'tax_code', 'employee_num', 'bus_own_type', 'b1_business_name', 'b1_address1', 'b1_city', 'b1_state', 'b1_zip', 'b1_contact_type', 'b1_full_address', 'b1_situs_city', 'b1_situs_state', 'b1_situs_zip']

📊 Testing: k2nh-s5h5
   Name: Berkeley PD - Calls for Service
   ✅ ACCESSIBLE - 5 sample rows, 18 columns
   Columns: ['caseno', 'offense', 'eventdt', 'eventtm', 'cvlegend', 'cvdow', 'indbdate', 'block_location', 'blkaddr', 'city', 'state', ':@computed_region_b3wi_w8ix', ':@computed_region_fhmw_rucx', ':@computed_region_u3y2_d2ws', ':@computed_region_5

In [8]:
!ls berkeley_datasets.csv

berkeley_datasets.csv


In [5]:
# CELL 4: Functions for Data Fetching

def fetch_berkeley_data(dataset_name, limit=10000, filters=None):
    """
    Fetch data from Berkeley Open Data Portal
    
    Parameters:
    -----------
    dataset_name : str
        Name of dataset from DATASETS dict
    limit : int
        Maximum number of records to fetch
    filters : dict
        Optional filters (e.g., {'city': 'Berkeley'})
    
    Returns:
    --------
    pandas.DataFrame
    """
    try:
        dataset_id = DATASETS.get(dataset_name)
        if not dataset_id:
            raise ValueError(f"Unknown dataset: {dataset_name}")
        
        print(f"📥 Fetching {dataset_name} from Berkeley Open Data...")
        
        # Build query parameters
        params = {"$limit": limit}
        if filters:
            # Convert filters to SoQL WHERE clause
            where_clauses = [f"{k}='{v}'" for k, v in filters.items()]
            params["$where"] = " AND ".join(where_clauses)
        
        # Fetch data
        results = client.get(dataset_id, **params)
        
        # Convert to DataFrame
        df = pd.DataFrame.from_records(results)
        
        print(f"✅ Fetched {len(df)} records")
        return df
        
    except Exception as e:
        print(f"❌ Error fetching data: {e}")
        return None

print("✅ Functions newly defined")

✅ Functions newly defined


In [24]:
# CELL 5: Fetch Business Licenses Data 

# Fetch all business licenses
business_licenses = fetch_berkeley_data('business_licenses', limit=50000)

# Verify BL-005071 is there
if business_licenses is not None:
    bl_005071 = business_licenses[business_licenses['recordid'] == 'BL-005071']
    if len(bl_005071) > 0:
        print("✅ BL-005071 found!")
        print(bl_005071)
    else:
        print("❌ BL-005071 not in data")

if business_licenses is not None:
    # Display basic info
    print("\n📊 Dataset Info:")
    print(f"Shape: {business_licenses.shape}")
    print(f"\nColumns: {business_licenses.columns.tolist()}")
    print(f"\nFirst few records:")
    display(business_licenses.head())
    
    # Check for location data
    if 'location' in business_licenses.columns:
        print("\n✅ Location data available for mapping")
    else:
        print("\n⚠️ No location column found")

📥 Fetching business_licenses from Berkeley Open Data...
✅ Fetched 11560 records
✅ BL-005071 found!
                apn   recordid        busdesc b1_per_sub_type  \
8055  052 157301400  BL-005071  OPTICAL STORE    Retail Trade   

                  dba                          naics tax_code employee_num  \
8055  FOCAL POINT INC  446130 - Optical Goods Stores        R            6   

     bus_own_type b1_business_name          b1_address1   b1_city b1_state  \
8055  Corporation  FOCAL POINT INC  2700 RYDIN RD STE A  RICHMOND       CA   

         b1_zip b1_contact_type b1_full_address b1_situs_city b1_situs_state  \
8055  948045800  Business Owner  2638 ASHBY AVE      BERKELEY             CA   

     b1_situs_zip b1_address2  
8055        94705         NaN  

📊 Dataset Info:
Shape: (11560, 20)

Columns: ['apn', 'recordid', 'busdesc', 'b1_per_sub_type', 'dba', 'naics', 'tax_code', 'employee_num', 'bus_own_type', 'b1_business_name', 'b1_address1', 'b1_city', 'b1_state', 'b1_zip', 'b1_con

,apn,recordid,busdesc,b1_per_sub_type,dba,naics,tax_code,employee_num,bus_own_type,b1_business_name,b1_address1,b1_city,b1_state,b1_zip,b1_contact_type,b1_full_address,b1_situs_city,b1_situs_state,b1_situs_zip,b1_address2
0,056 194500402,BL-027898,CLOTHING MFG,Manufacturing,BRYN WALKER,315239,M,46,Corporation,BRYN WALKER,2331 4TH ST,BERKELEY,CA,94710,Business Owner,2331 FOURTH ST,BERKELEY,CA,94710,NaN
1,056 193601900,BL-041509,CONSULTING ENGINEER,Professional SemiProfessional,SPARLING SCOTT,541330 - Engineering Services,P,0,NaN,SPARLING SCOTT,937 DWIGHT WAY,BERKELEY,CA,94710,Business Owner,937 DWIGHT WAY,BERKELEY,CA,94710,NaN
2,ZZZZZZZZZZZZZ,BL-004608,COFFEE,Wholesale Trade,FARMER BROS CO,424490,W,1,Corporation,FARMER BROS CO,14501 NORTH FWY,FORT WORTH,TX,76177 330,Business Owner,0 VARIOUS,BERKELEY,CA,94704,NaN
3,056 198304001,BL-037977,FAST FOOD RESTAURANT,Retail Trade,JACK IN THE BOX,722211,R,16,Corporation,JACK IN THE BOX,2197 SAN PABLO AVE,BERKELEY,CA,94702,Business Owner,2197 SAN PABLO AVE,BERKELEY,CA,94702,NaN
4,ZZZZZZZZZZZZZ,BL-031448,CONCRETE CONTRACTOR,Construction or Contractor,BRODERSON CONCRETE,"238190 - Other Foundation, Structure, and Buil...",C,0,Sole Ownership,BRODERSON CONCRETE,1616 AQUA VISTA RD,RICHMOND,CA,94805-2029,Business Owner,0 VARIOUS,BERKELEY,CA,94704,NaN



⚠️ No location column found


In [25]:
# CELL 5.1: Fetch Building Permits Data 

# Fetch all building permits..typed
building_permits = fetch_berkeley_data('building_permits', limit=50000)

if building_permits is not None:
    # Display basic info
    print("\n📊 Dataset Info:")
    print(f"Shape: {building_permits.shape}")
    print(f"\nColumns: {building_permits.columns.tolist()}")
    print(f"\nFirst few records:")
    display(building_permits.head())
    
    # Check for location data
    if 'location' in building_permits.columns:
        print("\n✅ Location data available for mapping")
    else:
        print("\n⚠️ No location column found")

📥 Fetching building_permits from Berkeley Open Data...
❌ Error fetching data: 403 Client Error: Forbidden


In [28]:
# Perplexity tries again
import requests
import pandas as pd
import sqlite3

DOMAIN = "data.cityofberkeley.info"
DATASET_ID = "5vy5-rwja"  # Berkeley building energy / BESO data

url = f"https://{DOMAIN}/resource/{DATASET_ID}.csv"
params = {
    "$limit": 50000,  # increase or implement paging if needed
}

resp = requests.get(url, params=params)  # no headers, no auth
resp.raise_for_status()

csv_bytes = resp.content  # raw bytes
df = pd.read_csv(pd.io.common.BytesIO(csv_bytes))

# Write to SQLite for Datasette
conn = sqlite3.connect("berkeley_energy_use.db")
df.to_sql("building_energy", conn, if_exists="replace", index=False)
conn.close()


In [43]:
# Perplexity Three: find dataset id's - creates a 73 row frame
import requests
import pandas as pd

DOMAIN = "data.cityofberkeley.info"
# q = "permit" # OR zoning OR housing OR demolition OR occupancy"

url = "https://api.us.socrata.com/api/catalog/v1"
params = {
    "domains": DOMAIN,
    "search_context": DOMAIN,
   # "q": q,
    "limit": 1000,
}

resp = requests.get(url, params=params)
print("Status:", resp.status_code)
print("URL:", resp.url)
print("First 500 chars of body:")
print(resp.text[:500])      # see what you actually got

resp.raise_for_status()
j = resp.json()

print("\nTop-level JSON keys:", list(j.keys()))
print("Result count:", j.get("resultSetSize"))
print("Results type:", type(j.get("results")))
print("First result (raw):")

if j.get("results"):
        print(j["results"][0])
else:
    print("No results")

rows = []
for item in j.get("results", []):
    res = item.get("resource", {}) or {}
    view = item.get("view", {}) or {}
    rows.append({
        "name": res.get("name") or view.get("name"),
        "description": res.get("description") or view.get("description"),
        "dataset_id": res.get("id"),
        "type": res.get("type"),
        "link": view.get("htmlPageUrl"),
        "api_url": f"https://{DOMAIN}/resource/{res.get('id')}.csv" if res.get("id") else None,
        "tags": ",".join(view.get("tags") or []),
        "categories": ",".join(view.get("category_tags") or []),
    })

catalog_df = pd.DataFrame(rows)
catalog_df



Status: 200
URL: https://api.us.socrata.com/api/catalog/v1?domains=data.cityofberkeley.info&search_context=data.cityofberkeley.info&limit=1000
First 500 chars of body:
{
  "results" :
    [
      {
        "resource" :
          {
            "name" : "Business Licenses",
            "id" : "rwnf-bu3w",
            "resource_name" : null,
            "parent_fxf" : [],
            "description" : "Registered businesses in the City of Berkeley with business license records",
            "attribution" : "City of Berkeley Finance Department",
            "attribution_link" : "http://www.ci.berkeley.ca.us/businesslicense/",
            "contact_email" : null,
    

Top-level JSON keys: ['results', 'resultSetSize', 'timings', 'warnings']
Result count: 73
Results type: <class 'list'>
First result (raw):
{'resource': {'name': 'Business Licenses', 'id': 'rwnf-bu3w', 'resource_name': None, 'parent_fxf': [], 'description': 'Registered businesses in the City of Berkeley with business license reco

,name,description,dataset_id,type,link,api_url,tags,categories
0,Business Licenses,Registered businesses in the City of Berkeley ...,rwnf-bu3w,dataset,None,https://data.cityofberkeley.info/resource/rwnf...,,
1,Berkeley PD - Calls for Service,Calls for service (not criminal reports) withi...,k2nh-s5h5,dataset,None,https://data.cityofberkeley.info/resource/k2nh...,,
2,Parcels,Parcel Polygon GIS database for the City of Be...,bhxd-e6up,map,None,https://data.cityofberkeley.info/resource/bhxd...,,
3,Taxable Square Footage,This dataset of taxable square footage per pro...,9a47-nj4i,dataset,None,https://data.cityofberkeley.info/resource/9a47...,,
4,BESO Building Energy Data and Compliance Statu...,This dataset contains the compliance status an...,5vy5-rwja,dataset,None,https://data.cityofberkeley.info/resource/5vy5...,,
5,"Berkeley PD - Stop Data (October 1, 2020 - Pre...",Berkeley PD transitioned to RIPA stop data col...,ysvs-bcge,dataset,None,https://data.cityofberkeley.info/resource/ysvs...,,
6,Sanitary Sewer Mainlines,Lines representing the approximate location of...,8qse-2t2u,map,None,https://data.cityofberkeley.info/resource/8qse...,,
7,Land Boundary,City of Berkeley land area polygon used for ge...,6is2-y2ia,map,None,https://data.cityofberkeley.info/resource/6is2...,,
8,Zoning Districts,Polygons illustrating zoning districts in the ...,2dtu-vge3,map,None,https://data.cityofberkeley.info/resource/2dtu...,,
9,Streets Network,Street centerline network for the City of Berk...,hqnk-qfhq,map,None,https://data.cityofberkeley.info/resource/hqnk...,,


In [38]:
# broad Perplexity search
params = {
    "domains": DOMAIN,
    "search_context": DOMAIN,
    "limit": 100,
}

resp = requests.get(url, params=params)
resp.raise_for_status()
j = resp.json()
print("ResultSetSize:", j.get("resultSetSize"))


ResultSetSize: 73


In [ ]:
# Perplexity fixed 9:11
import requests
import pandas as pd

DOMAIN = "data.cityofberkeley.info"
# q = "permit OR zoning OR housing OR demolition OR occupancy"

url = "https://api.us.socrata.com/api/catalog/v1"
params = {
    "domains": DOMAIN,
    "search_context": DOMAIN,
    # "q": q,
    "limit": 1000,
}

resp = requests.get(url, params=params)
print("Status:", resp.status_code)
print("URL:", resp.url)
print("First 400 chars of body:")
print(resp.text[:400])

resp.raise_for_status()
j = resp.json()

print("\nTop-level JSON keys:", list(j.keys()))
print("resultSetSize:", j.get("resultSetSize"))
print("Has results?", bool(j.get("results")))
if j.get("results"):
    print("First result (raw):")
    print(j["results"][0])

rows = []
for item in j.get("results", []):
    res = item.get("resource", {}) or {}
    view = item.get("view", {}) or {}
    rows.append({
        "name": res.get("name") or view.get("name"),
        "description": res.get("description") or view.get("description"),
        "dataset_id": res.get("id"),
        "type": res.get("type"),
        "link": view.get("htmlPageUrl"),
        "api_url": f"https://{DOMAIN}/resource/{res.get('id')}.csv" if res.get("id") else None,
        "tags": ",".join(view.get("tags") or []),
        "categories": ",".join(view.get("category_tags") or []),
    })

catalog_df = pd.DataFrame(rows)
print("\ncatalog_df shape:", catalog_df.shape)
catalog_df.head()


Status: 200
URL: https://api.us.socrata.com/api/catalog/v1?domains=data.cityofberkeley.info&search_context=data.cityofberkeley.info&q=permit+OR+zoning+OR+housing+OR+demolition+OR+occupancy&limit=1000
First 400 chars of body:
{
  "results" : [],
  "resultSetSize" : 0,
  "timings" : { "serviceMillis" : 33, "searchMillis" : [ 20, 6 ] },
  "warnings" : []
}

Top-level JSON keys: ['results', 'resultSetSize', 'timings', 'warnings']
resultSetSize: 0
Has results? False

catalog_df shape: (0, 0)


""


# new break point: 2025-1-7: Add new cells after me

## Below me are older cells that may still work

## Summary

This notebook demonstrates:
- ✅ Fetching data from Berkeley Open Data API
- ✅ Data cleaning and processing with Pandas
- ✅ Exporting to CSV, JSON, and GeoJSON
- ✅ Loading into SQLite for Datasette
- ✅ SQL queries for analysis
- ✅ Visualizations with Seaborn and Plotly
- ✅ Interactive maps

### Next Steps:
1. Add more datasets from Berkeley Open Data
2. Create custom dashboards
3. Schedule automated updates
4. Integrate with OSMnx for spatial analysis
5. Build predictive models with TensorFlow